# Notebook 4 - Layer Sweep

This notebook finds where the transferable signal is strongest in Phi-2. It trains each probe at every transformer layer and evaluates out-of-distribution grouped accuracy.

This is the notebook that justifies using `LAYER_INDEX = 18` in the final transfer matrix.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

RESULTS_DIR = PROJECT_ROOT / "results"
ACTIVATION_CACHE_DIR = PROJECT_ROOT / "data" / "activations"
RESULTS_DIR.mkdir(exist_ok=True)
ACTIVATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


## Build Datasets

The same dataset collection is used across notebooks so the results are comparable. Activation extraction is cached, so rerunning the sweep reuses the same hidden states.


In [ ]:
from lie_detector_llm.datasets import DEFAULT_DATASET_NAMES, build_dataset_collection

DATASET_NAMES = DEFAULT_DATASET_NAMES
MAX_GROUPS = 50

collection = build_dataset_collection(
    dataset_names=DATASET_NAMES,
    max_groups=MAX_GROUPS,
    seed=0,
)

display(collection.summary())


## Configure the Sweep

Phi-2 has 32 transformer layers. The sweep trains `DIM`, `LAT`, `LR`, and `PCA-G` at every layer. For each layer and probe, the model trains on `dbpedia_14` and evaluates on all datasets.


In [ ]:
from lie_detector_llm.experiment import DEFAULT_MODEL, PROBE_METHODS

MODEL_NAME = DEFAULT_MODEL
TRAIN_DATASET = "dbpedia_14"
ACTIVATION_BATCH_SIZE = 2
MAX_LENGTH = 512
LOAD_IN_4BIT = False

print("Model:", MODEL_NAME)
print("Training dataset:", TRAIN_DATASET)
print("Probe methods:", PROBE_METHODS)


In [ ]:
from lie_detector_llm.experiment import run_layer_method_sweep

sweep = run_layer_method_sweep(
    collection=collection,
    train_dataset_name=TRAIN_DATASET,
    probe_methods=PROBE_METHODS,
    model_name=MODEL_NAME,
    activation_batch_size=ACTIVATION_BATCH_SIZE,
    max_length=MAX_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    show_progress=True,
    activation_cache_dir=ACTIVATION_CACHE_DIR,
)

sweep.results.to_csv(RESULTS_DIR / "phi2_layer_method_sweep.csv", index=False)
display(sweep.results.head())


## Out-of-Distribution Accuracy by Layer

This is the main layer figure. The y-axis is grouped accuracy averaged across evaluation datasets other than the training dataset. Higher is better.

The saved Phi-2 run finds that `DIM` and `PCA-G` peak at layer 18 with mean OOD accuracy around 0.833.


In [ ]:
import matplotlib.pyplot as plt
from lie_detector_llm.plotting import plot_layer_method_sweep

fig, ax = plot_layer_method_sweep(
    sweep.results,
    eval_type="out_of_distribution",
    title=f"Phi-2 OOD truth-probe accuracy by layer, train={TRAIN_DATASET}",
)
fig.savefig(RESULTS_DIR / "phi2_layer_sweep_ood.png", dpi=160, bbox_inches="tight")
plt.show()


## In-Distribution Reference

This second plot uses held-out groups from `dbpedia_14`. In-distribution accuracy is useful as a sanity check, but the out-of-distribution plot is the main evidence for transfer.


In [ ]:
fig, ax = plot_layer_method_sweep(
    sweep.results,
    eval_type="in_distribution",
    title=f"Phi-2 in-distribution accuracy by layer, train={TRAIN_DATASET}",
)
fig.savefig(RESULTS_DIR / "phi2_layer_sweep_id.png", dpi=160, bbox_inches="tight")
plt.show()


## Best Layers

The first table finds the best probe-layer combinations. The second table averages across probes to find a single robust layer.


In [ ]:
ood = sweep.results[sweep.results["eval_type"] == "out_of_distribution"]

best_probe_layers = (
    oud.groupby(["probe_method", "layer"], as_index=False)["grouped_accuracy"]
    .mean()
    .sort_values("grouped_accuracy", ascending=False)
)

display(best_probe_layers.head(12))

best_layers_overall = (
    oud.groupby("layer", as_index=False)["grouped_accuracy"]
    .mean()
    .sort_values("grouped_accuracy", ascending=False)
)

display(best_layers_overall.head(10))


## Interpretation

The result is coherent with the original paper at a qualitative level: early layers are weak, and mid-to-late layers transfer better. For the final matrix, we use layer 18 because it is the best layer for the strongest individual probes (`DIM` and `PCA-G`).
